# Integrated seismic interpretation prototype

This notebook combines a 2D U-Net for fault detection and a section-based DeconvNet for facies classification. See the repository README for how this prototype differs from the final thesis experiments.


## Workflow

The two models independently generate overlays for the same seismic section.

1. Read datasets from the configured local folder or mounted Google Drive.
2. Load a compatible fault checkpoint or train the 2D U-Net on Wu FaultSeg3D.
3. Load a compatible facies checkpoint or train the DeconvNet on Alaudah F3 using focal Tversky loss and rare-class oversampling. Apply monotonic stratigraphic ordering during inference.
4. Predict faults and facies on the available SEG-Y sections.
5. Toggle the two overlays independently in the interactive viewer.

Training can be computationally expensive. Enable `QUICK_TEST` for a shorter trial or reuse compatible checkpoints with `FORCE_RETRAIN=False`.


## 0. Environment


In [ ]:
import sys, os, glob, struct, time, csv, copy, zipfile
from pathlib import Path

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

HAVE_SEGYIO = False
try:
    import segyio  # noqa
    HAVE_SEGYIO = True
except Exception:
    HAVE_SEGYIO = False   # Use the built-in NumPy SEG-Y reader if segyio is unavailable

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, CosineAnnealingWarmRestarts

SEED = 42
import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"IN_COLAB={IN_COLAB}  segyio={HAVE_SEGYIO}")
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE.type == 'cuda':
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    try:
        from google.colab import output as _coutput
        _coutput.enable_custom_widget_manager()
    except Exception:
        pass
else:
    print("Local environment. Skipping the Drive mount.")

## 1. Configuration

Set `SEISMIC_DATA_ROOT` to the input folder. The default is `data/` locally and `/content/drive/MyDrive/seismic-data` in Colab. Configure `SEGY_INPUTS` and `SEGY_DIR` for the sections you want to interpret. Checkpoints are stored under `<root>/seiswork/checkpoints/`.


In [ ]:
# =====================================================================
# 1. CONFIGURATION. Supply all input data locally.
# =====================================================================
# Expected layout under DATA_ROOT:
#   <root>/fault/fault_datas_wu/train/seis/*.dat        (Wu FaultSeg3D seismic cubes)
#   <root>/fault/fault_datas_wu/train/fault/*.dat       (Wu FaultSeg3D fault labels)
#   <root>/fault/fault_datas_wu/validation/seis/*.dat
#   <root>/fault/fault_datas_wu/validation/fault/*.dat
#   <root>/facies/alaudah_test_and_train/train_seismic.npy      (Alaudah F3)
#   <root>/facies/alaudah_test_and_train/train_labels.npy
#   <root>/facies/alaudah_test_and_train/test1_seismic.npy
#   <root>/facies/alaudah_test_and_train/test1_labels.npy
#   <root>/facies/f3_sections_test/*.sgy                    (field sections for interpretation)
DEFAULT_DATA_ROOT = '/content/drive/MyDrive/seismic-data' if IN_COLAB else 'data'
DATA_ROOT = Path(os.environ.get('SEISMIC_DATA_ROOT', DEFAULT_DATA_ROOT)).expanduser().resolve()
DRIVE = DATA_ROOT
WORK     = DATA_ROOT / 'seiswork'
CKPT_DIR = WORK / 'checkpoints'  # Checkpoints are stored in the working directory
CKPT_DIR.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

# ----- fault (Wu FaultSeg3D) -----
FAULT_ROOT = DATA_ROOT / 'fault' / 'fault_datas_wu'
F_TR_SEIS  = FAULT_ROOT / 'train'      / 'seis'
F_TR_FAULT = FAULT_ROOT / 'train'      / 'fault'
F_VA_SEIS  = FAULT_ROOT / 'validation' / 'seis'
F_VA_FAULT = FAULT_ROOT / 'validation' / 'fault'

# ----- facies (Alaudah F3) -----
FACIES_ROOT = DATA_ROOT / 'facies' / 'alaudah_test_and_train'
FC_TR_SEIS = FACIES_ROOT / 'train_seismic.npy'
FC_TR_LBL  = FACIES_ROOT / 'train_labels.npy'
FC_T1_SEIS = FACIES_ROOT / 'test1_seismic.npy'
FC_T1_LBL  = FACIES_ROOT / 'test1_labels.npy'
FC_T2_SEIS = FACIES_ROOT / 'test2_seismic.npy'   # Optional input
FC_T2_LBL  = FACIES_ROOT / 'test2_labels.npy'
if not Path(FC_T2_SEIS).exists(): FC_T2_SEIS = None; FC_T2_LBL = None

# ----- Checkpoint paths used for training and reuse -----
# Use checkpoints compatible with the model definitions below.
CKPT_FAULT  = CKPT_DIR / 'fault_unet2d.pt'
CKPT_FACIES = CKPT_DIR / 'facies_deconvnet.pt'   # Prototype facies model

# ----- Field SEG-Y sections. Missing files are skipped. -----
SEGY_DIR = DATA_ROOT / 'facies' / 'f3_sections_test'
SEGY_INPUTS = [
    SEGY_DIR / 'Seismic_data_Extract_250.sgy',
    SEGY_DIR / 'Seismic_data_Extract_270.sgy',
    SEGY_DIR / 'Seismic_data_Extract_304.sgy',
    SEGY_DIR / 'Seismic_data_Extract_457.sgy',
    SEGY_DIR / 'Seismic_data_Extract_521.sgy',
]
SEGY_INPUTS = [p for p in SEGY_INPUTS if p.exists()]  # Keep existing files only


# ----- Classes and colors -----
N_CLASSES    = 6
FACIES_NAMES = ['Upper N.S.', 'Middle N.S.', 'Lower N.S.', 'Rijnland/Chalk', 'Scruff', 'Zechstein']
FACIES_CMAP  = ListedColormap(['#4575b4', '#91bfdb', '#e0f3f8', '#fee090', '#fc8d59', '#d73027'])

# ----- Fault model and display settings -----
FAULT_THR = 0.9; FAULT_COLOR = '#ff1744'; FAULT_BASE = 32; CUBE_DIM = (128, 128, 128)

# ----- Training -----
FORCE_RETRAIN = False        # Retrain both models even when checkpoints exist
QUICK_TEST    = False        # Run a shorter workflow trial with subsampling
EPOCHS_FAULT  = 50 if not QUICK_TEST else 2
EPOCHS_FACIES = 30 if not QUICK_TEST else 2
BATCH_FAULT   = 16
BATCH_FACIES  = 8
NUM_WORKERS   = 2 if IN_COLAB else 0
USE_AMP       = True

print('=== Input data status ===')
for p, tag in [(F_TR_SEIS,'fault train/seis'), (F_TR_FAULT,'fault train/fault'),
               (F_VA_SEIS,'fault val/seis'), (F_VA_FAULT,'fault val/fault'),
               (FC_TR_SEIS,'facies train_seismic'), (FC_TR_LBL,'facies train_labels'),
               (FC_T1_SEIS,'facies test1_seismic'), (FC_T1_LBL,'facies test1_labels')]:
    extra = ''
    if Path(p).is_dir():
        extra = ' [%d .dat]' % len(list(Path(p).glob('*.dat')))
    print(f"  {tag:22s}: {'OK' if Path(p).exists() else 'MISSING':6s}{extra}  {p}")
print('Fault ckpt :', 'OK' if CKPT_FAULT.exists() else 'missing', '->', CKPT_FAULT)
print('Facies ckpt:', 'OK' if CKPT_FACIES.exists() else 'missing', '->', CKPT_FACIES)
print('SEG-Y input files available:', sum(Path(s).exists() for s in SEGY_INPUTS), '/', len(SEGY_INPUTS))


## 2. Data preparation

Place the extracted datasets under the configured input folder. If the fault training labels are still in `train/fault.zip`, this step extracts them when `train/fault/` is missing.


In [ ]:
def _safe_extract(zip_path, dest, skip_if=None):
    dest = Path(dest)
    if skip_if and Path(skip_if).exists():
        print(f"  already extracted → {skip_if}"); return
    if not Path(zip_path).exists():
        print(f"  ✗ not found: {zip_path}"); return
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    print(f"  extracted: {Path(zip_path).name} → {dest}")

# Check the extracted datasets
print("Fault data:")
print(f"  train/seis  : {'FOUND ✓' if F_TR_SEIS.exists()  else 'MISSING ✗'}  → {F_TR_SEIS}")
print(f"  val/seis    : {'FOUND ✓' if F_VA_SEIS.exists()  else 'MISSING ✗'}  → {F_VA_SEIS}")
print(f"  val/fault   : {'FOUND ✓' if F_VA_FAULT.exists() else 'MISSING ✗'}  → {F_VA_FAULT}")

# Extract train/fault.zip if the label folder is missing
_nested = FAULT_ROOT / 'train' / 'fault.zip'
if _nested.exists() and not F_TR_FAULT.exists():
    _safe_extract(_nested, FAULT_ROOT / 'train', skip_if=F_TR_FAULT)
elif F_TR_FAULT.exists():
    print(f"  train/fault : FOUND ✓  → {F_TR_FAULT}")
else:
    print(f"  train/fault : MISSING ✗  → {F_TR_FAULT}  (fault.zip is also missing)")

print("\nFacies data:")
print(f"  train_seismic: {'FOUND ✓' if FC_TR_SEIS.exists() else 'MISSING ✗'}  → {FC_TR_SEIS}")
print(f"  train_labels : {'FOUND ✓' if FC_TR_LBL.exists()  else 'MISSING ✗'}  → {FC_TR_LBL}")
print(f"  test1_seismic: {'FOUND ✓' if FC_T1_SEIS.exists() else 'MISSING ✗'}  → {FC_T1_SEIS}")
print(f"  test1_labels : {'FOUND ✓' if FC_T1_LBL.exists()  else 'MISSING ✗'}  → {FC_T1_LBL}")

print("\n— Detected paths —")
for nm, p in [('fault train/seis', F_TR_SEIS), ('fault train/fault', F_TR_FAULT),
              ('fault val/seis',   F_VA_SEIS),  ('fault val/fault',   F_VA_FAULT),
              ('facies train_seismic', FC_TR_SEIS), ('facies train_labels', FC_TR_LBL)]:
    if p and Path(p).exists():
        n = len(list(Path(p).glob('*.dat'))) if Path(p).is_dir() else 'FOUND'
        print(f"  {nm:22s}: {p}  [{n}]")
    else:
        print(f"  {nm:22s}: MISSING ✗")

## 3. Shared input format

Load SEG-Y, NumPy, or raw binary data into the common `(n_time, n_trace)` format.


In [ ]:
def _read_segy_numpy(path):
    with open(path, 'rb') as f:
        f.seek(3200); binh = f.read(400); body = f.read()
    u16 = lambda b, o: struct.unpack('>H', b[o:o+2])[0]
    ns  = u16(binh, 20); fmt = u16(binh, 24)
    if ns == 0: ns = u16(body[:240], 114)
    fmtsize = {1:4,5:4,2:4,3:2,8:1}.get(fmt, 4)
    bpt = 240 + ns*fmtsize; ntr = len(body)//bpt
    sec = np.empty((ntr, ns), dtype=np.float32)
    if fmt == 5:
        for i in range(ntr):
            off = i*bpt+240; sec[i] = struct.unpack('>%df'%ns, body[off:off+ns*4])
    elif fmt == 1:
        for i in range(ntr):
            off = i*bpt+240
            raw = np.frombuffer(body[off:off+ns*4], dtype='>u4').astype(np.uint32)
            sign=(raw>>31)&1; expo=(raw>>24)&0x7f; mant=(raw&0x00ffffff).astype(np.float64)/float(1<<24)
            sec[i] = ((1-2*sign)*mant*16.0**(expo-64)).astype(np.float32)
    else:
        raise ValueError(f"Unsupported SEG-Y format: {fmt}")
    return sec.T, {'ns': ns, 'ntr': ntr, 'fmt': fmt}

def read_segy(path):
    path = str(path)
    if HAVE_SEGYIO:
        with segyio.open(path, 'r', ignore_geometry=True) as f:
            data = np.stack([f.trace[i] for i in range(f.tracecount)])
        return data.T.astype(np.float32), {'ns': data.shape[1], 'ntr': data.shape[0]}
    return _read_segy_numpy(path)

def load_section(path, npy_inline=None, dat_shape=None):
    p = Path(path); ext = p.suffix.lower()
    if ext in ('.sgy', '.segy'):
        sec, _ = read_segy(p)
    elif ext == '.npy':
        arr = np.load(p, mmap_mode='r')
        if arr.ndim == 3:
            i = arr.shape[0]//2 if npy_inline is None else npy_inline
            sec = np.array(arr[i], dtype=np.float32).T
        elif arr.ndim == 2:
            sec = np.array(arr, dtype=np.float32)
        else:
            raise ValueError(f".npy ndim={arr.ndim}")
    elif ext == '.dat':
        if dat_shape is None: raise ValueError("dat_shape is required for .dat files")
        a = np.fromfile(str(p), dtype=np.float32).reshape(dat_shape)
        sec = np.transpose(a) if a.ndim == 2 else a[a.shape[0]//2].T
    else:
        raise ValueError(f"Unknown extension: {ext}")
    return sec.astype(np.float32)

def collect_inputs():
    files = [Path(x) for x in SEGY_INPUTS if Path(x).exists()]
    if not files and Path(SEGY_DIR).exists():
        for ext in ('*.sgy', '*.segy'):
            files += sorted(Path(SEGY_DIR).glob(ext))
    return files

INPUT_FILES = collect_inputs()
print(f"{len(INPUT_FILES)} test sections:")
for f in INPUT_FILES: print("  -", f.name)
if INPUT_FILES:
    d = load_section(INPUT_FILES[0])
    print(f"\nExample '{INPUT_FILES[0].name}': {d.shape} (time,trace) amp=[{d.min():.0f},{d.max():.0f}]")

## 4. Fault detection with a 2D U-Net

The fault model combines binary cross-entropy and Tversky loss. At inference it processes overlapping tiles with flip-based test-time augmentation and blends the results into a probability map.


In [ ]:
# -- Fault inference uses training normalization by default. The threshold is 0.50. --
FAULT_THR=0.50; FAULT_USE_AGC=False; FAULT_AGC_WIN=64
FAULT_USE_CLIP=False; FAULT_CLIP_PCT=99.5; FAULT_TTA='flip4'; FAULT_SMOOTH=0.0
FAULT_CLEAN_DEFAULT=False; FAULT_MIN_AREA=40; FAULT_MIN_ELONG=2.5
# -- Fault training uses multi-axis slicing and augmentation to improve transfer to field data. --
FAULT_MULTI_AXIS=True
FAULT_AUG_GAIN=0.30; FAULT_AUG_GAMMA=0.25; FAULT_AUG_NOISE=0.10; FAULT_AUG_FLIPV=True
# -- Reuse the existing fault checkpoint when available.
#    Set the flag to True to retrain the fault model.
FAULT_FORCE_RETRAIN=False

class FaultDoubleConv(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(i,o,3,padding=1,bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
            nn.Conv2d(o,o,3,padding=1,bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class FaultUNet2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=FAULT_BASE):
        super().__init__()
        self.enc1=FaultDoubleConv(in_ch,base); self.enc2=FaultDoubleConv(base,base*2)
        self.enc3=FaultDoubleConv(base*2,base*4); self.enc4=FaultDoubleConv(base*4,base*8)
        self.bott=FaultDoubleConv(base*8,base*16); self.pool=nn.MaxPool2d(2)
        self.up4=nn.ConvTranspose2d(base*16,base*8,2,2); self.up3=nn.ConvTranspose2d(base*8,base*4,2,2)
        self.up2=nn.ConvTranspose2d(base*4,base*2,2,2); self.up1=nn.ConvTranspose2d(base*2,base,2,2)
        self.dec4=FaultDoubleConv(base*16,base*8); self.dec3=FaultDoubleConv(base*8,base*4)
        self.dec2=FaultDoubleConv(base*4,base*2); self.dec1=FaultDoubleConv(base*2,base)
        self.out=nn.Conv2d(base,out_ch,1)
    def forward(self, x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1)); e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3))
        b=self.bott(self.pool(e4))
        d4=self.dec4(torch.cat([self.up4(b),e4],1)); d3=self.dec3(torch.cat([self.up3(d4),e3],1))
        d2=self.dec2(torch.cat([self.up2(d3),e2],1)); d1=self.dec1(torch.cat([self.up1(d2),e1],1))
        return self.out(d1)

def _load_state(ckpt_path):
    obj = torch.load(str(ckpt_path), map_location=DEVICE, weights_only=True)
    if isinstance(obj, dict):
        for k in ('model','model_state_dict','state_dict'):
            if k in obj: return obj[k], obj
        return obj, obj
    return obj.state_dict(), {}

# ---- Load Wu fault data ----
def fault_load_cube(path, shape=CUBE_DIM, normalize=False):
    a = np.fromfile(str(path), dtype=np.float32).reshape(shape)
    a = np.transpose(a)
    if normalize: a = (a - a.mean())/(a.std()+1e-8)
    return a.astype(np.float32)

def fault_collect_ids(seis_dir, fault_dir):
    s = {p.stem for p in Path(seis_dir).glob('*.dat')}
    fset = {p.stem for p in Path(fault_dir).glob('*.dat')}
    return sorted(s & fset, key=lambda z:int(z))

def fault_load_all(ids, seis_dir, fault_dir):
    n=len(ids); X=np.empty((n,*CUBE_DIM),np.float32); Y=np.empty((n,*CUBE_DIM),np.float32)
    for i,idx in enumerate(ids):
        X[i]=fault_load_cube(Path(seis_dir)/f'{idx}.dat', normalize=True)
        Y[i]=fault_load_cube(Path(fault_dir)/f'{idx}.dat', normalize=False)
    return X,Y

class FaultInlineDS(Dataset):
    def __init__(self, X, Y, augment=False):
        self.X=X; self.Y=Y; self.augment=augment
        nc=X.shape[0]
        # Slice each cube along all three axes to increase orientation coverage.
        self.axes=(0,1,2) if (augment and globals().get('FAULT_MULTI_AXIS',True)) else (2,)
        self.plan=[]   # (cube, axis, slice_index)
        for c in range(nc):
            for ax in self.axes:
                for j in range(CUBE_DIM[ax]):
                    self.plan.append((c,ax,j))
    def __len__(self): return len(self.plan)
    def _slice(self, arr, c, ax, j):
        if ax==0: return arr[c,j,:,:]
        if ax==1: return arr[c,:,j,:]
        return arr[c,:,:,j]
    def __getitem__(self, idx):
        c,ax,j=self.plan[idx]
        x=np.array(self._slice(self.X,c,ax,j),np.float32)
        y=np.array(self._slice(self.Y,c,ax,j),np.float32)
        if self.augment:
            if np.random.rand()<0.5:                       # horizontal flip
                x=x[:,::-1].copy(); y=y[:,::-1].copy()
            if globals().get('FAULT_AUG_FLIPV',True) and np.random.rand()<0.5:  # vertical flip
                x=x[::-1,:].copy(); y=y[::-1,:].copy()
            g=globals()
            if g.get('FAULT_AUG_GAIN',0)>0:                # amplitude scaling jitter
                x=x*np.float32(1.0+np.random.uniform(-g['FAULT_AUG_GAIN'],g['FAULT_AUG_GAIN']))
            if g.get('FAULT_AUG_GAMMA',0)>0:               # sign-preserving contrast and gamma jitter
                gm=np.float32(1.0+np.random.uniform(-g['FAULT_AUG_GAMMA'],g['FAULT_AUG_GAMMA']))
                x=np.sign(x)*(np.abs(x)**gm)
            if g.get('FAULT_AUG_NOISE',0)>0:               # additive noise
                x=x+np.random.normal(0,g['FAULT_AUG_NOISE'],x.shape).astype(np.float32)
            x=np.ascontiguousarray(x,np.float32); y=np.ascontiguousarray(y,np.float32)
        return torch.from_numpy(x).unsqueeze(0), torch.from_numpy(y).unsqueeze(0)

class FaultComboLoss(nn.Module):
    def __init__(self, pos_weight, alpha=0.7, beta=0.3, bce_w=0.5, tv_w=0.5, smooth=1e-7):
        super().__init__()
        self.a=alpha; self.b=beta; self.bw=bce_w; self.tw=tv_w; self.s=smooth
        self.bce=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    def forward(self, logits, target):
        bce=self.bce(logits,target)
        p=torch.sigmoid(logits).reshape(-1); t=target.reshape(-1)
        TP=(p*t).sum(); FN=(t*(1-p)).sum(); FP=((1-t)*p).sum()
        tv=(TP+self.s)/(TP+self.a*FN+self.b*FP+self.s)
        return self.bw*bce + self.tw*(1-tv)

def fault_train_model(model):
    assert F_TR_SEIS and F_TR_FAULT, "Fault training data is missing (train/seis, train/fault)."
    tr_ids = fault_collect_ids(F_TR_SEIS, F_TR_FAULT)
    if not tr_ids:
        raise RuntimeError('Fault training data is missing. Check for .dat files under %s and %s. '
                           'Place the Wu FaultSeg3D files there before running training.'
                           % (F_TR_SEIS, F_TR_FAULT))
    if F_VA_SEIS and F_VA_FAULT:
        va_ids = fault_collect_ids(F_VA_SEIS, F_VA_FAULT); va_seis, va_fault = F_VA_SEIS, F_VA_FAULT
    else:  # Hold out training cubes if no validation paths are configured
        k=max(1,int(len(tr_ids)*0.1)); va_ids=tr_ids[:k]; tr_ids=tr_ids[k:]; va_seis,va_fault=F_TR_SEIS,F_TR_FAULT
    if QUICK_TEST: tr_ids=tr_ids[:8]; va_ids=va_ids[:4]
    print(f"Fault training: {len(tr_ids)} train + {len(va_ids)} validation cubes loading...")
    Xtr,Ytr=fault_load_all(tr_ids,F_TR_SEIS,F_TR_FAULT)
    Xva,Yva=fault_load_all(va_ids,va_seis,va_fault)
    ratio=float(Ytr.mean()); pw=torch.tensor([np.sqrt((1-ratio)/max(ratio,1e-6))],device=DEVICE)
    print(f"  fault%={ratio*100:.2f}  pos_weight(sqrt)={pw.item():.2f}")
    trl=DataLoader(FaultInlineDS(Xtr,Ytr,True),batch_size=BATCH_FAULT,shuffle=True,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'))
    val=DataLoader(FaultInlineDS(Xva,Yva,False),batch_size=BATCH_FAULT,shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'))
    crit=FaultComboLoss(pw).to(DEVICE)
    opt=Adam(model.parameters(),lr=1e-4,weight_decay=1e-4)
    sched=CosineAnnealingLR(opt,T_max=EPOCHS_FAULT,eta_min=1e-6)
    scaler=torch.amp.GradScaler('cuda',enabled=(USE_AMP and DEVICE.type=='cuda'))
    @torch.no_grad()
    def _eval():
        model.eval(); TP=FP=FN=0.0
        for x,y in val:
            x=x.to(DEVICE);y=y.to(DEVICE)
            with torch.amp.autocast('cuda',enabled=(USE_AMP and DEVICE.type=='cuda')):
                pr=(torch.sigmoid(model(x))>0.5).float()
            t=(y>0.5).float(); TP+=(pr*t).sum().item(); FP+=(pr*(1-t)).sum().item(); FN+=((1-pr)*t).sum().item()
        return (TP+1e-7)/(TP+FP+FN+1e-7)
    best=-1; patience=10; left=patience
    hist={'loss':[],'acc':[],'val_iou':[]}
    for ep in range(1,EPOCHS_FAULT+1):
        model.train(); run=0; racc=0; nb=0
        for x,y in trl:
            x=x.to(DEVICE);y=y.to(DEVICE); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda',enabled=(USE_AMP and DEVICE.type=='cuda')):
                logits=model(x); loss=crit(logits,y)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update()
            with torch.no_grad():
                racc+=(((torch.sigmoid(logits)>0.5).float()==(y>0.5).float()).float().mean()).item()
            run+=loss.item(); nb+=1
        sched.step(); iou=_eval()
        hist['loss'].append(run/nb); hist['acc'].append(racc/nb); hist['val_iou'].append(iou)
        flag=''
        if iou>best:
            best=iou; left=patience
            torch.save({'model':model.state_dict(),'epoch':ep,'val_iou':iou}, CKPT_FAULT); flag=' ✓'
        else: left-=1
        print(f"  ep{ep:02d} loss={run/nb:.4f} valIoU@0.5={iou:.4f} pat={left}{flag}")
        if left<=0: print("  early stop"); break
    globals()['FAULT_HISTORY']=hist
    sd,_=_load_state(CKPT_FAULT); model.load_state_dict(sd); model.eval()
    print(f"✓ Fault training complete. Best valIoU@0.5={best:.4f} → {CKPT_FAULT}")

# ---- Train or load ----
FAULT_HISTORY=None
_FAULT_FR = FORCE_RETRAIN or globals().get('FAULT_FORCE_RETRAIN', False)
fault_model = FaultUNet2D().to(DEVICE); fault_ready=False
if CKPT_FAULT.exists() and not _FAULT_FR:
    sd,_=_load_state(CKPT_FAULT); fault_model.load_state_dict(sd); fault_model.eval(); fault_ready=True
    print("✓ Loaded fault model from checkpoint:", CKPT_FAULT)
elif F_TR_SEIS and F_TR_FAULT:
    fault_train_model(fault_model); fault_ready=True
else:
    print("✗ Skipping the fault layer because no checkpoint or training data is available.")

def _pad(a,m,mode):
    h,w=a.shape; th=((h+m-1)//m)*m; tw=((w+m-1)//m)*m
    pt,pl=(th-h)//2,(tw-w)//2; pb,pr=th-h-pt,tw-w-pl
    return np.pad(a,((pt,pb),(pl,pr)),mode=mode),(pt,pb,pl,pr)

def _agc(sec, win=FAULT_AGC_WIN):
    """Apply vertical sliding-window gain control to compensate for amplitude decay with depth."""
    from scipy.ndimage import uniform_filter1d
    rms=np.sqrt(uniform_filter1d(sec.astype(np.float32)**2, size=int(win), axis=0, mode='nearest'))+1e-6
    return sec/rms

def _fault_norm(sec):
    # By default, gain control and clipping are disabled to match training normalization.
    s=sec.astype(np.float32)
    if FAULT_USE_AGC: s=_agc(s)
    if FAULT_USE_CLIP:
        c=np.percentile(np.abs(s), FAULT_CLIP_PCT)+1e-8
        s=np.clip(s, -c, c)
    return (s-s.mean())/(s.std()+1e-8)

def _fault_tta_forward(x):
    pr=torch.sigmoid(fault_model(x))
    mode=FAULT_TTA
    if mode=='none': return pr
    outs=[pr, torch.flip(torch.sigmoid(fault_model(torch.flip(x,dims=[-1]))),dims=[-1])]
    if mode=='flip4':
        outs.append(torch.flip(torch.sigmoid(fault_model(torch.flip(x,dims=[-2]))),dims=[-2]))
        outs.append(torch.flip(torch.sigmoid(fault_model(torch.flip(x,dims=[-1,-2]))),dims=[-1,-2]))
    return sum(outs)/len(outs)

@torch.no_grad()
def fault_predict(section_tt, tile=128, overlap=64):
    """Predict with overlapping 128-pixel tiles, four-way flip augmentation, and Hann-weighted blending."""
    s=_fault_norm(section_tt).astype(np.float32); H,W=s.shape
    step=max(1,tile-overlap)
    wy=np.hanning(tile) if tile>1 else np.ones(1); wy=np.maximum(wy,1e-3)
    win=(wy[:,None]*wy[None,:]).astype(np.float32)
    Hp=max(tile, ((H-1)//step)*step+tile); Wp=max(tile, ((W-1)//step)*step+tile)
    sp=np.zeros((Hp,Wp),np.float32); sp[:H,:W]=s
    acc=np.zeros((Hp,Wp),np.float32); wsum=np.zeros((Hp,Wp),np.float32)
    ys=sorted(set(list(range(0,Hp-tile+1,step))+[Hp-tile]))
    xs=sorted(set(list(range(0,Wp-tile+1,step))+[Wp-tile]))
    for y0 in ys:
        for x0 in xs:
            patch=sp[y0:y0+tile, x0:x0+tile]
            x=torch.from_numpy(patch).unsqueeze(0).unsqueeze(0).to(DEVICE)
            pr=_fault_tta_forward(x).squeeze().float().cpu().numpy()
            acc[y0:y0+tile, x0:x0+tile]+=pr*win
            wsum[y0:y0+tile, x0:x0+tile]+=win
    prob=(acc/np.maximum(wsum,1e-6))[:H,:W]
    if FAULT_SMOOTH and FAULT_SMOOTH>0:
        from scipy.ndimage import gaussian_filter
        prob=gaussian_filter(prob, FAULT_SMOOTH)
    return prob

def fault_clean_mask(prob, thr=None, min_area=None, min_elong=None):
    """Threshold the probability map and remove small or round connected components."""
    from scipy.ndimage import label
    thr=FAULT_THR if thr is None else thr
    min_area=FAULT_MIN_AREA if min_area is None else min_area
    min_elong=FAULT_MIN_ELONG if min_elong is None else min_elong
    m=prob>thr; lab,n=label(m); keep=np.zeros_like(m)
    for i in range(1,n+1):
        ys,xs=np.where(lab==i); a=len(ys)
        if a<min_area: continue
        if min_elong and min_elong>1 and a>=4:
            ys=ys-ys.mean(); xs=xs-xs.mean()
            cov=np.array([[ (ys*ys).mean(),(ys*xs).mean()],[(ys*xs).mean(),(xs*xs).mean()]])
            ev=np.linalg.eigvalsh(cov); ev=np.clip(ev,1e-6,None)
            if (ev[1]/ev[0])**0.5 < min_elong: continue   # Discard round speckles
        keep[lab==i]=True
    return keep

print("Fault model ready:", fault_ready)

## 5. Facies classification with a section-based DeconvNet

The model uses class-weighted cross-entropy, focal Tversky loss, and rare-class oversampling. Inference applies smoothing and enforces monotonic stratigraphic ordering along each trace.


In [ ]:
# =====================================================================
# 5. FACIES MODEL - Section-based DeconvNet (Alaudah 2019) + Focal-Tversky
#    + rare-class oversampling and monotonic stratigraphic ordering.
#    Downstream interface: facies_predict / facies_predict_clean / facies_model /
#    facies_ready / FACIES_HISTORY / FACIES_TRAIN_DEPTH / facies_train_model.
# =====================================================================
FACIES_MEAN      = 0.000941                 # Alaudah training mean for normalization
FACIES_PRIOR     = np.array([0.285, 0.118, 0.484, 0.066, 0.032, 0.015], np.float32)
FACIES_COH_SIGMA = (1.0, 6.0)               # Gaussian smoothing (depth, lateral)
FACIES_PRIOR_TAU = 0.0                      # Rare-class rebalancing (0 disables it)

class section_deconvnet(nn.Module):
    """Section-based DeconvNet of Alaudah et al. (2019): a VGG-16 encoder with
    index-recording max-pooling, a fully-convolutional bottleneck (fc6/fc7), and a
    symmetric decoder that restores resolution through max-unpooling. The module
    and layer names follow the published reference verbatim so that previously
    trained weights load without modification. Ceil-mode pooling together with
    explicit unpool output sizes makes the network agnostic to the input shape,
    so whole (depth x trace) sections of arbitrary size can be processed."""
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.unpool = nn.MaxUnpool2d(2, stride=2)

        def cbr(i, o):    # conv -> batch-norm -> ReLU
            return [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True)]

        def dcbr(i, o):   # transposed-conv -> batch-norm -> ReLU
            return [nn.ConvTranspose2d(i, o, 3, stride=1, padding=1),
                    nn.BatchNorm2d(o), nn.ReLU(inplace=True)]

        pool = lambda: nn.MaxPool2d(2, stride=2, return_indices=True, ceil_mode=True)
        # Encoder (VGG-16 configuration)
        self.conv_block1 = nn.Sequential(*cbr(1, 64),   *cbr(64, 64),   pool())
        self.conv_block2 = nn.Sequential(*cbr(64, 128),  *cbr(128, 128), pool())
        self.conv_block3 = nn.Sequential(*cbr(128, 256), *cbr(256, 256), *cbr(256, 256), pool())
        self.conv_block4 = nn.Sequential(*cbr(256, 512), *cbr(512, 512), *cbr(512, 512), pool())
        self.conv_block5 = nn.Sequential(*cbr(512, 512), *cbr(512, 512), *cbr(512, 512), pool())
        # Fully-convolutional bottleneck
        self.conv_block6 = nn.Sequential(nn.Conv2d(512, 4096, 3), nn.BatchNorm2d(4096), nn.ReLU(inplace=True))
        self.conv_block7 = nn.Sequential(nn.Conv2d(4096, 4096, 1), nn.BatchNorm2d(4096), nn.ReLU(inplace=True))
        self.deconv_block8 = nn.Sequential(nn.ConvTranspose2d(4096, 512, 3, stride=1),
                                           nn.BatchNorm2d(512), nn.ReLU(inplace=True))
        # Decoder
        self.deconv_block10 = nn.Sequential(*dcbr(512, 512), *dcbr(512, 512), *dcbr(512, 512))
        self.deconv_block12 = nn.Sequential(*dcbr(512, 512), *dcbr(512, 512), *dcbr(512, 256))
        self.deconv_block14 = nn.Sequential(*dcbr(256, 256), *dcbr(256, 256), *dcbr(256, 128))
        self.deconv_block16 = nn.Sequential(*dcbr(128, 128), *dcbr(128, 64))
        self.deconv_block18 = nn.Sequential(*dcbr(64, 64), *dcbr(64, 64))
        self.seg_score19 = nn.Sequential(nn.Conv2d(64, n_classes, 1))

    def forward(self, x):
        s0 = x.size()
        c1, i1 = self.conv_block1(x); s1 = c1.size()
        c2, i2 = self.conv_block2(c1); s2 = c2.size()
        c3, i3 = self.conv_block3(c2); s3 = c3.size()
        c4, i4 = self.conv_block4(c3); s4 = c4.size()
        c5, i5 = self.conv_block5(c4)
        d = self.deconv_block8(self.conv_block7(self.conv_block6(c5)))
        d = self.deconv_block10(self.unpool(d, i5, output_size=s4))
        d = self.deconv_block12(self.unpool(d, i4, output_size=s3))
        d = self.deconv_block14(self.unpool(d, i3, output_size=s2))
        d = self.deconv_block16(self.unpool(d, i2, output_size=s1))
        d = self.deconv_block18(self.unpool(d, i1, output_size=s0))
        return self.seg_score19(d)


# ---- Loss: class-weighted cross-entropy + Focal-Tversky --------------------
def ce_mean(logits, target, weight):
    return F.cross_entropy(logits, target, weight=weight, ignore_index=255, reduction='mean')


def focal_tversky(logits, target, alpha=0.3, beta=0.7, gamma=4.0 / 3.0):
    """Multi-class Focal-Tversky loss (Abraham & Khan, 2019). beta > alpha
    penalises false negatives, which directly raises the recall of the rare,
    deep facies (chalk, scruff, zechstein)."""
    C = logits.shape[1]
    valid = (target != 255); t = target.clone(); t[~valid] = 0
    oh = F.one_hot(t, C).permute(0, 3, 1, 2).float() * valid.unsqueeze(1)
    p = F.softmax(logits, 1) * valid.unsqueeze(1)
    dims = (0, 2, 3)
    TP = (p * oh).sum(dims); FP = (p * (1 - oh)).sum(dims); FN = ((1 - p) * oh).sum(dims)
    tv = (TP + 1e-6) / (TP + alpha * FP + beta * FN + 1e-6)
    return ((1 - tv) ** gamma).mean()


# ---- Section sampling with rare-class oversampling -------------------------
def _facies_section_names(labels):
    n_il, n_xl = labels.shape[0], labels.shape[1]
    return ['i_%d' % k for k in range(n_il)] + ['x_%d' % k for k in range(n_xl)]


def _oversample(names, labels):
    """Duplicate sections rich in the rare facies (classes 3, 4, 5) up to 3x."""
    out = []
    for nm in names:
        d, k = nm.split('_'); k = int(k)
        sl = labels[k] if d == 'i' else labels[:, k]
        frac = float(np.mean(np.isin(sl, [3, 4, 5])))
        out += [nm] * (1 + int(frac > 0.05) + int(frac > 0.15))
    return out


class FaciesSectionDS(Dataset):
    """One whole inline/crossline section per item; mean-subtracted, transposed
    to (depth, trace) as required by the network."""
    def __init__(self, seismic, labels, names):
        self.s, self.l, self.names = seismic, labels, names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        d, k = self.names[i].split('_'); k = int(k)
        im = self.s[k] if d == 'i' else self.s[:, k]
        lb = self.l[k] if d == 'i' else self.l[:, k]
        im = (im.astype(np.float32) - FACIES_MEAN).T
        lb = lb.astype(np.int64).T
        return (torch.from_numpy(np.ascontiguousarray(im))[None].float(),
                torch.from_numpy(np.ascontiguousarray(lb)).long())


class _HomogeneousBatch(torch.utils.data.Sampler):
    """Yield batches that are entirely inline OR entirely crossline so that all
    sections in a batch share the same width (the two orientations differ)."""
    def __init__(self, names, batch_size, shuffle=True):
        import random
        self._random = random
        self.i = [k for k, n in enumerate(names) if n[0] == 'i']
        self.x = [k for k, n in enumerate(names) if n[0] == 'x']
        self.bs, self.shuffle = batch_size, shuffle

    def __iter__(self):
        i, x = self.i[:], self.x[:]
        if self.shuffle:
            self._random.shuffle(i); self._random.shuffle(x)
        batches = [i[k:k + self.bs] for k in range(0, len(i), self.bs)] + \
                  [x[k:k + self.bs] for k in range(0, len(x), self.bs)]
        if self.shuffle:
            self._random.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return (len(self.i) + self.bs - 1) // self.bs + (len(self.x) + self.bs - 1) // self.bs


def _class_weights(labels):
    sub = labels[::4, ::4, ::2]
    cnt = np.array([(sub == c).sum() for c in range(N_CLASSES)], np.float64) + 1
    w = cnt.sum() / (N_CLASSES * cnt)
    return torch.from_numpy((w / w.mean()).astype(np.float32)).to(DEVICE)


def _facies_running_miou(model, loader):
    from numpy import bincount
    cm = np.zeros((N_CLASSES, N_CLASSES), np.int64)
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            p = model(x.to(DEVICE)).argmax(1).cpu().numpy().ravel(); g = y.numpy().ravel()
            cm += bincount(g * N_CLASSES + p, minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)
    iou = np.diag(cm) / (cm.sum(1) + cm.sum(0) - np.diag(cm) + 1e-9)
    return np.nanmean(iou), iou


def facies_train_model(model):
    """Train the facies DeconvNet (weighted CE + Focal-Tversky, rare-class
    oversampling, lateral-flip and additive-noise augmentation)."""
    seis = np.load(FC_TR_SEIS); labels = np.load(FC_TR_LBL)
    from sklearn.model_selection import train_test_split
    names = _facies_section_names(labels)
    tr, va = train_test_split(names, test_size=0.1, shuffle=True, random_state=SEED)
    tr_os = _oversample(tr, labels)
    tr_ld = DataLoader(FaciesSectionDS(seis, labels, tr_os),
                       batch_sampler=_HomogeneousBatch(tr_os, BATCH_FACIES, True))
    va_ld = DataLoader(FaciesSectionDS(seis, labels, va),
                       batch_sampler=_HomogeneousBatch(va, BATCH_FACIES, False))
    cw = _class_weights(labels)
    opt = torch.optim.Adam(model.parameters(), amsgrad=True)
    best = -1.0
    hist = {'loss': [], 'val_miou': []}
    for ep in range(1, EPOCHS_FACIES + 1):
        model.train(); run = 0.0; nb = 0
        for x, y in tr_ld:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if np.random.rand() < 0.5:                      # lateral (trace) flip
                x, y = torch.flip(x, [-1]), torch.flip(y, [-1])
            x = x + torch.randn_like(x) * 0.02              # additive noise
            opt.zero_grad()
            out = model(x)
            loss = ce_mean(out, y, cw) + focal_tversky(out, y)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1); opt.step()
            run += loss.item(); nb += 1
        miou, iou = _facies_running_miou(model, va_ld)
        hist['loss'].append(run / max(nb, 1)); hist['val_miou'].append(miou)
        flag = ''
        if miou >= best:
            best = miou
            CKPT_FACIES.parent.mkdir(parents=True, exist_ok=True)
            torch.save({'model_state_dict': model.state_dict(), 'val_miou': miou}, CKPT_FACIES); flag = ' *'
        print('  facies ep%02d val_mIoU=%.4f | %s%s'
              % (ep, miou, ' '.join('%.2f' % v for v in iou), flag))
    _sd, _ = _load_state(CKPT_FACIES); model.load_state_dict(_sd); model.eval()
    globals()['FACIES_HISTORY'] = hist
    print(f'Facies training done. best val_mIoU={best:.4f}')


# ---- Inference: domain-matching + lateral smoothing + monotonic ordering ----
def _facies_norm_section(sec):
    """Scale a raw field section to the [-1, 1] training range, then subtract
    the training mean (consistent with the labelled cubes)."""
    c = np.percentile(np.abs(sec), 99) + 1e-8
    return (np.clip(sec / c, -1, 1).astype(np.float32) - FACIES_MEAN)


def _monotonic_order(prob):
    """Per-trace dynamic program that returns the most probable labelling whose
    class index is non-decreasing with depth, enforcing the F3 stratigraphic
    order (Upper -> Middle -> Lower N.S. -> Chalk -> Scruff -> Zechstein).
    `prob` has shape (C, depth, trace)."""
    C, D, W = prob.shape
    logL = np.log(prob + 1e-8)
    dp = logL[:, 0, :].copy()
    back = np.zeros((D, C, W), np.int8)
    for d in range(1, D):
        best = dp[0].copy(); arg = np.zeros(W, np.int64)
        pref = np.empty((C, W)); a = np.zeros((C, W), np.int64); pref[0] = dp[0]
        for c in range(1, C):
            better = dp[c] > best
            best = np.where(better, dp[c], best); arg = np.where(better, c, arg)
            pref[c], a[c] = best, arg
        dp = logL[:, d, :] + pref; back[d] = a
    lab = np.zeros((D, W), int); lab[D - 1] = dp.argmax(0)
    for d in range(D - 1, 0, -1):
        lab[d - 1] = back[d][lab[d], np.arange(W)]
    return lab


@torch.no_grad()
def facies_predict(section_tt, return_proba=False):
    """Infer a facies label map for a canonical (n_time, n_trace) section.

    Pipeline: (i) amplitude normalisation; (ii) vertical domain-matching that
    resamples the section to the training depth (FACIES_TRAIN_DEPTH); (iii)
    lateral-flip TTA; (iv) Gaussian smoothing of the class posteriors;
    (v) optional prior re-balancing; (vi) monotonic stratigraphic ordering;
    (vii) resampling of the label map back to the native depth.
    """
    from scipy.ndimage import gaussian_filter, zoom
    sec = _facies_norm_section(section_tt.T)            # (trace, depth)
    nt = sec.shape[1]
    if nt != FACIES_TRAIN_DEPTH:                        # domain-match vertical scale
        sec = zoom(sec, (1.0, FACIES_TRAIN_DEPTH / float(nt)), order=1).astype(np.float32)
    im = sec.T                                          # (depth, trace)
    x = torch.from_numpy(np.ascontiguousarray(im))[None, None].float().to(DEVICE)
    logits = facies_model(x)
    logits = 0.5 * (logits + torch.flip(facies_model(torch.flip(x, [-1])), [-1]))  # lateral TTA
    prob = F.softmax(logits, 1)[0].cpu().numpy()        # (C, depth, trace)
    if FACIES_PRIOR_TAU > 0:                            # rare-class re-balancing
        prob = prob / (FACIES_PRIOR[:, None, None] ** FACIES_PRIOR_TAU)
        prob /= prob.sum(0, keepdims=True)
    prob = np.stack([gaussian_filter(prob[c], FACIES_COH_SIGMA) for c in range(N_CLASSES)])
    prob /= prob.sum(0, keepdims=True)
    lab = _monotonic_order(prob)                        # (depth_255, trace)
    if nt != FACIES_TRAIN_DEPTH:
        lab = zoom(lab, (nt / float(FACIES_TRAIN_DEPTH), 1.0), order=0).astype(int)
        prob = zoom(prob, (1.0, nt / float(prob.shape[1]), 1.0), order=1).astype(np.float32)
    if return_proba:
        return lab, prob                                # (depth, trace), (C, depth, trace)
    return lab

# --- Reference depth used to resample input sections for inference ---
def _facies_train_depth():
    try:
        if FC_TR_SEIS and Path(FC_TR_SEIS).exists():
            return int(np.load(FC_TR_SEIS, mmap_mode='r').shape[-1])
    except Exception:
        pass
    return 255
FACIES_TRAIN_DEPTH = _facies_train_depth()
print('Facies training depth used for inference:', FACIES_TRAIN_DEPTH)

# --- Train or load the DeconvNet checkpoint ---
FACIES_HISTORY = None
FACIES_FORCE_RETRAIN = False
_FACIES_FR = FORCE_RETRAIN or FACIES_FORCE_RETRAIN
facies_model = section_deconvnet(N_CLASSES).to(DEVICE); facies_ready = False
if CKPT_FACIES.exists() and not _FACIES_FR:
    _sd, _ = _load_state(CKPT_FACIES); facies_model.load_state_dict(_sd); facies_model.eval(); facies_ready = True
    print('Loaded facies DeconvNet checkpoint:', CKPT_FACIES)
elif Path(FC_TR_SEIS).exists() and Path(FC_TR_LBL).exists():
    facies_train_model(facies_model); facies_ready = True
else:
    print('Skipping the facies layer because no checkpoint or training data is available.')

# --- facies_predict_clean returns labels (time, trace) and probabilities (classes, time, trace) ---
@torch.no_grad()
def facies_predict_clean(section_tt):
    return facies_predict(section_tt, return_proba=True)

def facies_clean(proba):
    return np.asarray(proba).argmax(0)

print('Facies model ready:', facies_ready)


## 6. Run both models independently on the input sections


In [ ]:
RESULTS = {}
for f in INPUT_FILES:
    sec=load_section(f)
    e={'section':sec,'fault_prob':None,'facies_lbl':None,'facies_proba':None}
    if fault_ready:  e['fault_prob']=fault_predict(sec)
    if facies_ready: e['facies_lbl'],e['facies_proba']=facies_predict_clean(sec)
    RESULTS[f.name]=e
    fp=e['fault_prob']; fl=e['facies_lbl']
    cov='—'
    if fp is not None:
        cov='cov%.3f@%.2f (after cleanup %.3f)'%((fp>FAULT_THR).mean(),FAULT_THR,fault_clean_mask(fp).mean())
    _dnote=''
    try:
        _td=int(globals().get('FACIES_TRAIN_DEPTH',0) or 0)
        if _td and sec.shape[0]!=_td: _dnote=f" [depth resampled from {sec.shape[0]} to {_td}]"
    except Exception: pass
    print(f"{f.name:34s} sec={sec.shape} fault={cov} "
          f"facies={np.unique(fl).tolist() if fl is not None else '—'}{_dnote}")
print(f"\n{len(RESULTS)} sections processed.")

## 7. Training curves

Plot the available training and validation history by epoch. A checkpoint loaded in this session has no recorded training history in memory.


In [ ]:
def _plot_hist(hist, title, metric_keys, metric_label):
    if not hist or not hist.get('loss'):
        print(f"{title}: no training history is available for this session."); return
    ep=np.arange(1,len(hist['loss'])+1)
    fig,ax=plt.subplots(1,3,figsize=(16,4))
    ax[0].plot(ep,hist['loss'],marker='o',color='#d62728')
    ax[0].set_title(f'{title} — loss'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].grid(alpha=.3)
    if hist.get('acc'):
        ax[1].plot(ep,hist['acc'],marker='o',color='#2ca02c')
    ax[1].set_title(f'{title} — training accuracy'); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy'); ax[1].grid(alpha=.3)
    for k in metric_keys:
        if k in hist and hist[k]: ax[2].plot(ep,hist[k],marker='o',label=k)
    ax[2].set_title(f'{title} — validation {metric_label}'); ax[2].set_xlabel('epoch')
    ax[2].set_ylabel(metric_label); ax[2].legend(); ax[2].grid(alpha=.3)
    plt.tight_layout(); plt.show()

_plot_hist(globals().get('FAULT_HISTORY'),  'Fault (2D U-Net)',     ['val_iou'],            'IoU@0.5')
_plot_hist(globals().get('FACIES_HISTORY'), 'Facies (DeconvNet)',['val_miou'],'mIoU')

## 8. Facies evaluation on Alaudah F3 test1

Report pixel accuracy, per-class recall, per-class IoU, and mean IoU on sampled inlines. Show a confusion matrix and a representative section with predictions, ground truth, entropy, and errors.


In [ ]:
def _metrics_from_cm(cm):
    eps=1e-12; tp=np.diag(cm).astype(np.float64)
    fp=cm.sum(0)-tp; fn=cm.sum(1)-tp
    acc=cm.trace()/(cm.sum()+eps)
    recall=tp/(cm.sum(1)+eps)
    iou=tp/(tp+fp+fn+eps)
    return acc, recall, iou

def facies_evaluate(max_inlines=40):
    if not facies_ready: print("The facies model is unavailable."); return
    if not (FC_T1_SEIS.is_file() and FC_T1_LBL.is_file()): print("Labeled facies test1 data is missing."); return
    seis=np.load(FC_T1_SEIS,mmap_mode='r'); lbl=np.load(FC_T1_LBL,mmap_mode='r')
    ninl = min(seis.shape[0], lbl.shape[0]); step=max(1,ninl//max_inlines); idxs=list(range(0,ninl,step))
    print(f"test cube: seis={seis.shape} lbl={lbl.shape}  inlines evaluated={len(idxs)}")
    cm=np.zeros((N_CLASSES,N_CLASSES),np.int64)
    for i in idxs:
        sec=np.array(seis[i],dtype=np.float32).T          # standard (time, trace) layout
        gt =np.array(lbl[i]).astype(np.int64).T
        pred,_=facies_predict_clean(sec)                  # post-processed prediction
        h=min(pred.shape[0],gt.shape[0]); w=min(pred.shape[1],gt.shape[1])
        p=pred[:h,:w].ravel(); g=gt[:h,:w].ravel()
        m=(g>=0)&(g<N_CLASSES)
        cm+=np.bincount(N_CLASSES*g[m]+p[m],minlength=N_CLASSES**2).reshape(N_CLASSES,N_CLASSES)
    acc,recall,iou=_metrics_from_cm(cm)
    print(f"\n=== Facies metrics ===")
    print(f"Overall pixel accuracy : {acc*100:.2f} %")
    print(f"Mean IoU          : {np.nanmean(iou)*100:.2f} %")
    print(f"{'Class':22s} {'Recall':>9s} {'IoU':>8s}")
    for c in range(N_CLASSES):
        print(f"{FACIES_NAMES[c]:22s} {recall[c]*100:8.2f}% {iou[c]*100:7.2f}%")
    # confusion matrix (row-normalized)
    cmn=cm/np.maximum(cm.sum(1,keepdims=True),1)
    fig,ax=plt.subplots(figsize=(6.5,5.5))
    im=ax.imshow(cmn,cmap='Blues',vmin=0,vmax=1)
    ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
    ax.set_xticklabels(FACIES_NAMES,rotation=45,ha='right',fontsize=8); ax.set_yticklabels(FACIES_NAMES,fontsize=8)
    ax.set_xlabel('Prediction'); ax.set_ylabel('Ground truth'); ax.set_title('Confusion matrix (row-normalized)')
    for r in range(N_CLASSES):
        for cc in range(N_CLASSES):
            ax.text(cc,r,f'{cmn[r,cc]:.2f}',ha='center',va='center',
                    color='white' if cmn[r,cc]>0.5 else 'black',fontsize=7)
    fig.colorbar(im,fraction=0.046); plt.tight_layout(); plt.show()
    # Example section with prediction, ground truth, entropy, and errors
    i0=idxs[len(idxs)//2]
    sec=np.array(seis[i0],dtype=np.float32).T; gt=np.array(lbl[i0]).astype(np.int64).T
    pred,proba=facies_predict_clean(sec)
    h=min(pred.shape[0],gt.shape[0]); w=min(pred.shape[1],gt.shape[1])
    pred=pred[:h,:w]; gt=gt[:h,:w]; sec=sec[:h,:w]; proba=proba[:,:h,:w]
    ent=-(proba*np.log(proba+1e-12)).sum(0)
    diff=(pred!=gt).astype(float)
    vmax=np.percentile(np.abs(sec),98)
    fig,ax=plt.subplots(4,1,figsize=(12,14))
    ax[0].imshow(sec,cmap='gray',aspect='auto',vmin=-vmax,vmax=vmax)
    ax[0].imshow(pred,cmap=FACIES_CMAP,aspect='auto',alpha=0.5,vmin=0,vmax=N_CLASSES-1,interpolation='nearest')
    ax[0].set_title(f'Predicted facies — inline {i0}')
    ax[1].imshow(sec,cmap='gray',aspect='auto',vmin=-vmax,vmax=vmax)
    ax[1].imshow(gt,cmap=FACIES_CMAP,aspect='auto',alpha=0.5,vmin=0,vmax=N_CLASSES-1,interpolation='nearest')
    ax[1].set_title('Ground truth')
    im2=ax[2].imshow(ent,cmap='jet',aspect='auto'); ax[2].set_title('Uncertainty (entropy)'); fig.colorbar(im2,ax=ax[2],fraction=0.025)
    ax[3].imshow(diff,cmap='gray',aspect='auto'); ax[3].set_title(f'Error map (white indicates errors, accuracy={100*(1-diff.mean()):.1f}%)')
    for a in ax: a.set_xlabel('trace'); a.set_ylabel('time')
    plt.tight_layout(); plt.show()
    return cm

_facies_cm = facies_evaluate()

## 9. Fault evaluation on Wu FaultSeg3D validation data

Report IoU, precision, recall, F1, and Dice on sampled inlines. A threshold sweep shows which probability cutoff gives the highest IoU on this sample.


In [ ]:
def fault_evaluate(thr=FAULT_THR, sec_step=8, sweep=True):
    if not fault_ready: print("The fault model is unavailable."); return
    if not (F_VA_SEIS and F_VA_FAULT): print("Labeled fault validation data is missing."); return
    ids=fault_collect_ids(F_VA_SEIS,F_VA_FAULT)
    if QUICK_TEST: ids=ids[:3]
    probs=[]; gts=[]
    for idx in ids:
        seis=fault_load_cube(Path(F_VA_SEIS)/f'{idx}.dat',normalize=True)
        flt =fault_load_cube(Path(F_VA_FAULT)/f'{idx}.dat',normalize=False)
        for j in range(0,seis.shape[-1],sec_step):
            probs.append(fault_predict(seis[:,:,j])); gts.append(flt[:,:,j]>0.5)
    P=np.stack(probs); G=np.stack(gts)
    print(f"sections evaluated={P.shape[0]}  ({len(ids)} cubes)")
    def _m(t):
        pr=P>t; tp=(pr&G).sum(); fp=(pr&~G).sum(); fn=(~pr&G).sum()
        iou=tp/(tp+fp+fn+1e-9); prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9)
        f1=2*prec*rec/(prec+rec+1e-9); dice=2*tp/(2*tp+fp+fn+1e-9)
        return iou,prec,rec,f1,dice
    iou,prec,rec,f1,dice=_m(thr)
    print(f"\n=== Fault metrics @ threshold={thr:.2f} ===")
    print(f"IoU={iou:.4f}  Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}  Dice={dice:.4f}")
    if sweep:
        ts=np.round(np.arange(0.20,0.81,0.05),2); ious=[_m(t)[0] for t in ts]
        bi=int(np.argmax(ious)); print(f"Best IoU={ious[bi]:.4f} @ threshold={ts[bi]:.2f}")
        fig,ax=plt.subplots(figsize=(7,4)); ax.plot(ts,ious,marker='o')
        ax.axvline(ts[bi],ls='--',color='r'); ax.set_xlabel('threshold'); ax.set_ylabel('IoU')
        ax.set_title('Fault threshold sweep'); ax.grid(alpha=.3); plt.tight_layout(); plt.show()
    # Example sections
    k=min(3,P.shape[0]); sel=np.linspace(0,P.shape[0]-1,k).astype(int)
    fig,ax=plt.subplots(k,2,figsize=(11,3.2*k)); ax=np.atleast_2d(ax)
    for r,s in enumerate(sel):
        ax[r,0].imshow(P[s],cmap='hot',aspect='auto',vmin=0,vmax=1); ax[r,0].set_title(f'predicted probability #{s}')
        ax[r,1].imshow(G[s],cmap='gray',aspect='auto'); ax[r,1].set_title('ground truth')
    plt.tight_layout(); plt.show()

fault_evaluate()

## 10. Static overlay preview


In [ ]:
def render_section(name, show_fault=True, show_facies=True,
                   fault_thr=FAULT_THR, fault_alpha=0.55, facies_alpha=0.45,
                   clean_fault=FAULT_CLEAN_DEFAULT, ax=None):
    e=RESULTS[name]; sec=e['section']
    if ax is None: fig,ax=plt.subplots(figsize=(13,6))
    vmax=np.percentile(np.abs(sec),98)
    ax.imshow(sec,cmap='gray',aspect='auto',vmin=-vmax,vmax=vmax)
    if show_facies and e['facies_lbl'] is not None:
        ax.imshow(e['facies_lbl'],cmap=FACIES_CMAP,aspect='auto',vmin=0,vmax=N_CLASSES-1,
                  alpha=facies_alpha,interpolation='nearest')
    if show_fault and e['fault_prob'] is not None:
        if clean_fault:
            mask=fault_clean_mask(e['fault_prob'], thr=fault_thr)
        else:
            mask=e['fault_prob']>fault_thr
        fm=np.ma.masked_where(~mask, np.ones_like(e['fault_prob']))
        ax.imshow(fm,cmap=ListedColormap([FAULT_COLOR]),aspect='auto',alpha=fault_alpha,
                  vmin=0,vmax=1,interpolation='nearest')
    L=[]
    if show_facies and e['facies_lbl'] is not None: L.append('facies')
    if show_fault and e['fault_prob'] is not None: L.append('fault')
    ax.set_title(f"{name}  [{' + '.join(L) if L else 'seismic'}]",fontsize=10)
    ax.set_xlabel('trace'); ax.set_ylabel('time/depth'); return ax

for name in RESULTS:
    fig,ax=plt.subplots(figsize=(13,6)); render_section(name,ax=ax)
    h=[plt.Rectangle((0,0),1,1,facecolor=FACIES_CMAP(i)) for i in range(N_CLASSES)]+[plt.Rectangle((0,0),1,1,facecolor=FAULT_COLOR)]
    fig.legend(h,FACIES_NAMES+['fault'],loc='lower center',ncol=N_CLASSES+1,bbox_to_anchor=(0.5,-0.05),fontsize=8)
    plt.tight_layout(); plt.show()

## 11. Interactive viewer

Toggle each overlay, adjust opacity and fault threshold, or apply fault mask cleanup.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
if RESULTS:
    name_dd=widgets.Dropdown(options=list(RESULTS.keys()),description='Section:')
    cb_fault=widgets.Checkbox(value=True,description='Fault layer')
    cb_facies=widgets.Checkbox(value=True,description='Facies layer')
    cb_clean=widgets.Checkbox(value=FAULT_CLEAN_DEFAULT,description='Clean fault mask')
    thr_sl=widgets.FloatSlider(value=FAULT_THR,min=0.05,max=0.95,step=0.05,description='Fault threshold',continuous_update=False)
    fa_sl=widgets.FloatSlider(value=0.55,min=0.1,max=1.0,step=0.05,description='Fault opacity',continuous_update=False)
    ca_sl=widgets.FloatSlider(value=0.45,min=0.1,max=1.0,step=0.05,description='Facies opacity',continuous_update=False)
    def _u(name,show_fault,show_facies,clean_fault,fault_thr,fault_alpha,facies_alpha):
        fig,ax=plt.subplots(figsize=(13,6))
        render_section(name,show_fault,show_facies,fault_thr,fault_alpha,facies_alpha,clean_fault=clean_fault,ax=ax)
        h=[plt.Rectangle((0,0),1,1,facecolor=FACIES_CMAP(i)) for i in range(N_CLASSES)]+[plt.Rectangle((0,0),1,1,facecolor=FAULT_COLOR)]
        fig.legend(h,FACIES_NAMES+['fault'],loc='lower center',ncol=N_CLASSES+1,bbox_to_anchor=(0.5,-0.05),fontsize=8)
        plt.tight_layout(); plt.show()
    ui=widgets.VBox([widgets.HBox([name_dd]),widgets.HBox([cb_fault,cb_facies,cb_clean]),widgets.HBox([thr_sl,fa_sl,ca_sl])])
    out=widgets.interactive_output(_u,{'name':name_dd,'show_fault':cb_fault,'show_facies':cb_facies,
        'clean_fault':cb_clean,'fault_thr':thr_sl,'fault_alpha':fa_sl,'facies_alpha':ca_sl})
    display(ui,out)
else:
    print("Run Section 6 first.")

## 12. Optional export

Save the predicted fault probabilities and facies labels as NumPy arrays.


In [ ]:
SAVE_DIR=DRIVE / 'integrated_outputs'; SAVE_DIR.mkdir(parents=True,exist_ok=True)
for name,e in RESULTS.items():
    stem=Path(name).stem
    if e['fault_prob'] is not None: np.save(SAVE_DIR/f'{stem}_fault_prob.npy',e['fault_prob'].astype(np.float32))
    if e['facies_lbl'] is not None: np.save(SAVE_DIR/f'{stem}_facies_label.npy',e['facies_lbl'].astype(np.uint8))
    fig,ax=plt.subplots(figsize=(13,6)); render_section(name,ax=ax)
    fig.savefig(SAVE_DIR/f'{stem}_overlay.png',dpi=140,bbox_inches='tight'); plt.close(fig)
print("Saved to →", SAVE_DIR.resolve())